# Fashion-MNIST · Red neuronal densa

Ejecuta las tres celdas de código en orden. Consulta la [metodología](../docs/metodologia.md). Los comentarios y cifras se calculan al ejecutar; las alertas no certifican calidad del modelo.


In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("fashion_mnist_classification")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from fashion_mnist_classification.alerts import configurar_notebook, mostrar_alertas
configurar_notebook()


In [2]:
from fashion_mnist_classification.flow import run_stage
metrics, comment = run_stage(PROJECT_ROOT, "modelado")
display(Markdown(comment))
import pandas as pd
display(pd.DataFrame(metrics["candidatos"]))


# Clasificación visual de prendas: modelado

Se selecciona una **MLP de 64 neuronas ocultas**, con **alpha 0.001**, por F1 macro de validación. No se reentrena con validación ni se consulta test para elegir. Es una red densa sobre píxeles aplanados, **no una CNN**; no incorpora convoluciones ni aumento de datos. Las alertas de convergencia deben revisarse, sin asumir que alcanzar el máximo de iteraciones implica convergencia.


,alpha,iteraciones,accuracy,f1_macro
0,0.0001,40,0.859667,0.861163
1,0.0010,40,0.860667,0.862351


## Alertas e informe

Revisa la descripción y acción sugerida de cada aviso.


In [3]:
mostrar_alertas(PROJECT_ROOT/"reports/alertas/04_modelado.json")
from fashion_mnist_classification.informe import generar_informe
informe = generar_informe(PROJECT_ROOT)
display(Markdown("[Informe integral](../reports/informe_final/informe_final.md)"))


Tipo,Veces,Qué significa,Qué revisar,Detalle original
ConvergenceWarning,2,El algoritmo terminó sin alcanzar el criterio de convergencia solicitado.,"Revisar escalamiento, tolerancia y número de iteraciones antes de interpretar el modelo.",Ver detalle y origenStochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.C:\Users\mauro\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785


[Informe integral](../reports/informe_final/informe_final.md)